In [1]:
# Install dependencies (run once)
!pip install torch transformers sentence-transformers faiss-cpu pypdf tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 93.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.6/330.6 kB 35.3 MB/s eta 0:00:00


In [2]:
import torch  # PyTorch backend
print(torch.cuda.is_available())  # True if GPU is available

True


In [3]:
# Download the PDF to the Colab file system
!wget -O /content/llm_ebook.pdf "https://www.amax.com/content/files/2024/03/llm-ebook-part1-1.pdf"

--2026-02-17 08:27:33--  https://www.amax.com/content/files/2024/03/llm-ebook-part1-1.pdf
Resolving www.amax.com (www.amax.com)... 151.101.3.7, 151.101.67.7, 151.101.131.7, ...
Connecting to www.amax.com (www.amax.com)|151.101.3.7|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://storage.ghost.io/c/35/17/35170502-dfe4-4f36-9612-bdc657f28241/content/files/2024/03/llm-ebook-part1-1.pdf [following]
--2026-02-17 08:27:33--  https://storage.ghost.io/c/35/17/35170502-dfe4-4f36-9612-bdc657f28241/content/files/2024/03/llm-ebook-part1-1.pdf
Resolving storage.ghost.io (storage.ghost.io)... 199.232.211.7, 199.232.215.7, 2a04:4e42:4e::775, ...
Connecting to storage.ghost.io (storage.ghost.io)|199.232.211.7|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1304748 (1.2M) [application/pdf]
Saving to: ‘/content/llm_ebook.pdf’

/content/llm_ebook. 100%[===================>]   1.24M  3.15MB/s    in 0.4s    

2026-02-17 08:27:34 (3

In [4]:
from pypdf import PdfReader  # PDF text extraction

def load_pdf_text(pdf_path):
    """Read all pages from a PDF and return concatenated text."""
    reader = PdfReader(pdf_path)
    full_text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            full_text += page_text + "\n"
    return full_text

pdf_path = "/content/llm_ebook.pdf"
document_text = load_pdf_text(pdf_path)

In [5]:
from transformers import AutoTokenizer  # Tokenizer for chunking

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

def chunk_text_by_tokens(text, chunk_size=512, overlap=80):
    """Split text into overlapping token chunks."""
    encoding = tokenizer(
        text,
        add_special_tokens=False,
        truncation=True,
        max_length=chunk_size,
        stride=overlap,
        return_overflowing_tokens=True,
    )
    chunk_texts = []
    for token_ids in encoding["input_ids"]:
        chunk_texts.append(tokenizer.decode(token_ids))
    return chunk_texts

text_chunks = chunk_text_by_tokens(document_text)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [6]:
from sentence_transformers import SentenceTransformer  # Embedding model

embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

chunk_embeddings = embedding_model.encode(
    text_chunks,
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=True,
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
import faiss  # Vector search
import numpy as np

embedding_dim = chunk_embeddings.shape[1]
chunk_embeddings_np = chunk_embeddings.cpu().numpy()
faiss.normalize_L2(chunk_embeddings_np)  # Normalize for cosine similarity with inner product

index = faiss.IndexFlatIP(embedding_dim)
index.add(chunk_embeddings_np)

In [8]:
def retrieve_top_k(query_text, top_k=10):
    """Embed the query and retrieve top-k similar chunks from FAISS."""
    query_embedding = embedding_model.encode([query_text], convert_to_tensor=True)
    query_embedding_np = query_embedding.cpu().numpy()
    faiss.normalize_L2(query_embedding_np)

    scores, indices = index.search(query_embedding_np, top_k)
    return indices[0], scores[0]

In [9]:
from sentence_transformers import CrossEncoder  # Cross-encoder reranker

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cuda")

def rerank_candidates(query_text, candidate_chunks):
    """Rerank retrieved chunks with a cross-encoder."""
    pairs = [(query_text, chunk) for chunk in candidate_chunks]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(candidate_chunks, scores), key=lambda x: x[1], reverse=True)
    return ranked

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [10]:
def search(query_text, retrieve_k=5, final_k=3):
    """Retrieve candidates with embeddings, then rerank with cross-encoder."""
    indices, _ = retrieve_top_k(query_text, retrieve_k)
    candidate_chunks = [text_chunks[i] for i in indices]

    ranked = rerank_candidates(query_text, candidate_chunks)
    return ranked[:final_k]

In [11]:
print(search("How do you define a large language model?"))

[('mainly white - collar workers. 5. large language models can generate inappropriate and harmful content. on that note, enterprises should keep in mind that llms are often trained on large corpora of internet texts, which may make them prone to generating toxic, biased, and otherwise inappropriate and harmful content. enterprises that want to build proprietary llms from scratch also need to address additional challenges, like whether they have sufficient computing power, storage, and datasets, expertise, and financial resources to develop, implement, and maintain the models. ways to build llms building large language models from scratch doesn ’ t always make sense, especially for enterprises whose core business is not related to ai or nlp technologies. since the process can be extremely time - consuming and resource - exhaustive, most enterprises are more likely to opt for customizing existing models to their needs. a beginner ’ s guide to large language models 22 customizing existing